In [23]:
import pandas as pd

df = pd.read_csv(
    'RetailCleaned.csv',
    header=None,
    dtype={0: 'string'},
    low_memory=False
)

df.columns = [
    'Invoice', 'StockCode', 'Description', 'Quantity',
    'InvoiceDate', 'Price', 'Customer_ID', 'Country', 'SourceSheet'
]

# Convert all object columns to strings
object_cols = df.select_dtypes(include=['object']).columns
df[object_cols] = df[object_cols].astype('string')

# Ensure required columns have the correct data types
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['Invoice'] = df['Invoice'].astype('string')
df['StockCode'] = df['StockCode'].astype('string')
df['Description'] = df['Description'].astype('string')
df['Country'] = df['Country'].astype('string')
df['SourceSheet'] = df['SourceSheet'].astype('string')

print(df.shape)
df.info()

(1033036, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1033036 entries, 0 to 1033035
Data columns (total 9 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1033036 non-null  string        
 1   StockCode    1033036 non-null  string        
 2   Description  1028761 non-null  string        
 3   Quantity     1033036 non-null  int64         
 4   InvoiceDate  1033036 non-null  datetime64[ns]
 5   Price        1033036 non-null  float64       
 6   Customer_ID  797885 non-null   float64       
 7   Country      1033036 non-null  string        
 8   SourceSheet  1033036 non-null  string        
dtypes: datetime64[ns](1), float64(2), int64(1), string(5)
memory usage: 70.9 MB


In [25]:
#FILTER AND CALCULATE RECENCY, FREQUENCY, MONETARY

rfm_data = df[(df['Customer_ID'].notna()) & (df['Quantity'] > 0)].copy()
snapshot_date = pd.to_datetime('2011-12-10')

rfm = rfm_data.groupby('Customer_ID').agg(
    Recency = ('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency = ('Invoice', 'nunique'),
    Monetary = ('Quantity', lambda x: (x * rfm_data.loc[x.index, 'Price']).sum())
).reset_index()

In [27]:
# SCORE CUSTOMERS AND ASSIGN BUSINESS SEGMENTS
rfm['R_Score'] = pd.qcut(rfm['Recency'], 5, labels=[5,4,3,2,1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[5,4,3,2,1]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 5, labels=[5,4,3,2,1]).astype(int)

def segment_customer(row):
    if row['R_Score'] >= 4 and row['F_Score'] >= 4 and row['M_Score'] >= 4:
        return 'Champions'
    elif row['R_Score'] >= 3 and row['F_Score'] >= 3:
        return 'Loyal'
    elif row['R_Score'] >= 2 and row['F_Score'] >= 3: 
        return 'At Risk'
    elif row['R_Score'] >= 4 and row['F_Score'] >= 2:
        return 'New'
    elif row['R_Score'] >= 3 and row['F_Score'] >= 2:
        return 'Nedds Attention'
    else:
        return 'Lost'

rfm['Segment'] = rfm.apply(segment_customer, axis=1)
rfm['Is_Churned'] = rfm['Recency'] > 90

In [29]:
# MERGE RFM RESULTS BACK ONTO TRANSACTIONS AND EXPORT
df = df[(df['Customer_ID'].notna()) & (df['Quantity'] > 0)].copy()
df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M').dt.to_timestamp()
df['Revenue'] = df['Quantity'] * df['Price']

df_final = df.merge(
    rfm[['Customer_ID', 'Recency', 'Frequency', 'Monetary', 'R_Score', 'F_Score', 'M_Score', 'Segment', 'Is_Churned']],
         on='Customer_ID', how='left'
)

df_final.to_excel('EcommerceDashboardData.xlsx', index=False)